# MedGemma 1.5 4B — Colab GPU Runner (Hybride)

Lanceur minimal : le code source reste versionné dans le repo Git ; Colab sert uniquement de banc GPU (**L4** pour la baseline, **A100 40 Go** pour QLoRA). Pensez à **déconnecter le runtime** dès la fin du run.

In [ ]:
# 1. Vérification GPU (bfloat16 requis) & montage Google Drive
import torch
from google.colab import drive

assert torch.cuda.is_available(), 'Aucun GPU détecté : choisir L4 ou A100.'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
bf16_ok = torch.cuda.is_bf16_supported()
print(f'GPU: {gpu_name} ({vram_gb:.1f} Go VRAM) | bfloat16 natif: {bf16_ok}')
assert bf16_ok, 'bfloat16 non supporté (éviter T4 pour MedGemma).'

drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/MedGemma_Dermato'

In [ ]:
# 2. Authentification Hugging Face via Colab Secrets (jamais de token en dur)
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
assert hf_token, 'Ajouter HF_TOKEN dans les Colab Secrets (icône Clé à gauche).'
login(token=hf_token, add_to_git_credential=False)
print('Authentification Hugging Face OK.')

In [ ]:
# 3. Clone/mise à jour du repo GitHub et installation en mode éditable avec dépendances GPU
import os

REPO_URL = 'https://github.com/sailwaelan-dotcom/medgemma-dermato.git'
REPO_DIR = '/content/medgemma-dermato'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only

%cd /content/medgemma-dermato
!pip install -q -e '.[train]'

In [ ]:
# 4. Smoke test cross-hospitalier (réutilise data/isic_smoke sur Drive si présent) + baseline zéro-shot
import os

SMOKE_DIR = '/content/drive/MyDrive/MedGemma_Dermato/data/isic_smoke'
if not os.path.exists(f'{SMOKE_DIR}/test.jsonl'):
    !python scripts/build_dataset.py --mode cross-hospital --commercial-only --max-items 150 --skip-exclude-fetch --output-dir {SMOKE_DIR}

!python scripts/run_baseline.py --manifest {SMOKE_DIR}/test.jsonl --image-dir /content/drive/MyDrive/MedGemma_Dermato/data/images --output-dir /content/drive/MyDrive/MedGemma_Dermato/outputs/baseline_smoke --limit 20

In [ ]:
# 5. Entraînement QLoRA (A100 40 Go) avec checkpoints persistés sur Drive
!python scripts/train_qlora.py --train-manifest /content/drive/MyDrive/MedGemma_Dermato/data/isic_smoke/train.jsonl --val-manifest /content/drive/MyDrive/MedGemma_Dermato/data/isic_smoke/val.jsonl --image-dir /content/drive/MyDrive/MedGemma_Dermato/data/images --output-dir /content/drive/MyDrive/MedGemma_Dermato/checkpoints/qlora_run1

# IMPORTANT : déconnecter le runtime immédiatement après la fin du run :
# from google.colab import runtime; runtime.unassign()